<a href="https://colab.research.google.com/github/yuvrajrajput/transformer-from-scratch/blob/main/Experimental_SLM_for_EN_to_FR_Translation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformer notebook (production pipeline)

**Legacy Sections 1–2 removed.** This notebook now focuses on one path: SentencePiece BPE, RoPE encoder–decoder (**`Seq2SeqTransformerV3`**), SDPA-backed attention where available, training with validation/checkpoints, and beam-search evaluation.

Earlier tutorial cells (GPT-style LM and word-split vocabulary) lived in prior copies if you need them.


In [ ]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [ ]:
# English–French pairs (Kaggle). Install once if needed.
try:
    import kagglehub
except ImportError:
    import subprocess

    subprocess.check_call(["pip", "install", "-q", "kagglehub"])
    import kagglehub

import os
import pandas as pd

path = kagglehub.dataset_download("dhruvildave/en-fr-translation-dataset")
print("Dataset path:", path)

files = os.listdir(path)
print("Files:", files)

df = pd.read_csv(os.path.join(path, files[0]), nrows=50000)
df = df.dropna()
df = df[df["en"].str.len() > 3]
df = df[df["fr"].str.len() > 3]
df = df[df["en"].str.len() < 100]
df = df[df["fr"].str.len() < 100]
df = df.reset_index(drop=True)
print("Rows after filtering:", len(df))


Using Colab cache for faster access to the 'en-fr-translation-dataset' dataset.
Dataset path: /kaggle/input/en-fr-translation-dataset
Files: ['en-fr.csv']
Rows after filtering: 16355


## Production training pipeline

- config-driven hyperparameters
- train/val/test split
- subword tokenizer (SentencePiece)
- Dataset + DataLoader with dynamic padding
- RoPE encoder–decoder + SDPA-backed attention
- dropout, grad clipping, scheduler warmup
- mixed precision training (AMP)
- checkpoint save/resume + early stopping
- validation each epoch
- beam-search decode + BLEU/chrF, perplexity, latency


In [ ]:
import os
import math
import json
import random
import tempfile
import csv
import time
import numpy as np
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

# Optional install helper
try:
    import sentencepiece as spm
except ImportError:
    import subprocess

    subprocess.check_call(["pip", "install", "-q", "sentencepiece"])
    import sentencepiece as spm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


config = {
    "embed_size": 256,
    "num_layers": 4,
    "heads": 8,
    "forward_expansion": 4,
    "dropout": 0.1,
    "max_len": 128,
    "batch_size": 32,
    "num_epochs": 50,
    "lr": 3e-4,
    "warmup_steps": 400,
    "grad_clip": 1.0,
    "vocab_size": 8000,
    "checkpoint_path": "s2s_best.pt",
    "metrics_path": "train_metrics.csv",
    "early_stopping_patience": 8,
}

print(json.dumps(config, indent=2))

{
  "embed_size": 256,
  "num_layers": 4,
  "heads": 8,
  "forward_expansion": 4,
  "dropout": 0.1,
  "max_len": 128,
  "batch_size": 32,
  "num_epochs": 50,
  "lr": 0.0003,
  "warmup_steps": 400,
  "grad_clip": 1.0,
  "vocab_size": 8000,
  "checkpoint_path": "s2s_best.pt",
  "metrics_path": "train_metrics.csv",
  "early_stopping_patience": 8
}


In [ ]:
# Split once and keep held-out data truly unseen
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=SEED, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED, shuffle=True)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

# Train SentencePiece on train split only
with tempfile.NamedTemporaryFile(mode="w", delete=False, encoding="utf-8") as f:
    corpus_file = f.name
    for _, row in train_df.iterrows():
        f.write(f"EN: {row['en']}\n")
        f.write(f"FR: {row['fr']}\n")

sp_prefix = "spm_enfr"
spm.SentencePieceTrainer.train(
    input=corpus_file,
    model_prefix=sp_prefix,
    vocab_size=config["vocab_size"],
    model_type="bpe",
    character_coverage=1.0,
    bos_id=1,
    eos_id=2,
    unk_id=0,
    pad_id=3,
)

sp = spm.SentencePieceProcessor(model_file=f"{sp_prefix}.model")
os.remove(corpus_file)

PAD_ID = sp.pad_id()
UNK_ID = sp.unk_id()
BOS_ID = sp.bos_id()
EOS_ID = sp.eos_id()

print("Special IDs =>", {"unk": UNK_ID, "bos": BOS_ID, "eos": EOS_ID, "pad": PAD_ID})

train=13084, val=1635, test=1636
Special IDs => {'unk': 0, 'bos': 1, 'eos': 2, 'pad': 3}


In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, frame, tokenizer):
        self.frame = frame
        self.sp = tokenizer

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        src_ids = self.sp.encode(row["en"], out_type=int)
        tgt_ids = self.sp.encode(row["fr"], out_type=int)
        return {
            "src": src_ids,
            "tgt": [BOS_ID] + tgt_ids + [EOS_ID],
        }


def collate_translation(batch, pad_id=PAD_ID, max_len=128):
    src_list = [item["src"][:max_len] for item in batch]
    tgt_list = [item["tgt"][:max_len] for item in batch]

    src_max = max(len(x) for x in src_list)
    tgt_max = max(len(x) for x in tgt_list)

    src_tensor = torch.full((len(batch), src_max), pad_id, dtype=torch.long)
    tgt_tensor = torch.full((len(batch), tgt_max), pad_id, dtype=torch.long)

    for i, ids in enumerate(src_list):
        src_tensor[i, : len(ids)] = torch.tensor(ids, dtype=torch.long)

    for i, ids in enumerate(tgt_list):
        tgt_tensor[i, : len(ids)] = torch.tensor(ids, dtype=torch.long)

    return src_tensor, tgt_tensor


train_ds = TranslationDataset(train_df, sp)
val_ds = TranslationDataset(val_df, sp)
test_ds = TranslationDataset(test_df, sp)

train_loader = DataLoader(
    train_ds,
    batch_size=config["batch_size"],
    shuffle=True,
    collate_fn=lambda b: collate_translation(b, max_len=config["max_len"]),
)
val_loader = DataLoader(
    val_ds,
    batch_size=config["batch_size"],
    shuffle=False,
    collate_fn=lambda b: collate_translation(b, max_len=config["max_len"]),
)
test_loader = DataLoader(
    test_ds,
    batch_size=config["batch_size"],
    shuffle=False,
    collate_fn=lambda b: collate_translation(b, max_len=config["max_len"]),
)

src_b, tgt_b = next(iter(train_loader))
print("Batch shapes =>", src_b.shape, tgt_b.shape)

Batch shapes => torch.Size([32, 25]) torch.Size([32, 30])


In [ ]:
# RoPE upgrade: modern positional handling on attention Q/K
class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_position=4096, base=10000.0):
        super().__init__()
        if dim % 2 != 0:
            raise ValueError("RoPE dimension must be even")
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_position, dtype=torch.float32)
        freqs = torch.einsum("i,j->ij", t, inv_freq)
        self.register_buffer("cos_cached", torch.cos(freqs), persistent=False)
        self.register_buffer("sin_cached", torch.sin(freqs), persistent=False)

    def get_cos_sin(self, seq_len, device, dtype, offset=0):
        cos = self.cos_cached[offset : offset + seq_len].to(device=device, dtype=dtype)
        sin = self.sin_cached[offset : offset + seq_len].to(device=device, dtype=dtype)
        return cos.unsqueeze(0).unsqueeze(2), sin.unsqueeze(0).unsqueeze(2)


def rotate_half(x):
    x1 = x[..., ::2]
    x2 = x[..., 1::2]
    return torch.stack((-x2, x1), dim=-1).flatten(-2)


def apply_rope(x, cos, sin):
    return (x * cos.repeat_interleave(2, dim=-1)) + (rotate_half(x) * sin.repeat_interleave(2, dim=-1))


class MultiHeadAttentionRoPE(nn.Module):
    def __init__(self, embed_size, heads, dropout=0.1, max_position=4096):
        super().__init__()
        if embed_size % heads != 0:
            raise ValueError("embed_size must be divisible by heads")
        self.heads = heads
        self.head_dim = embed_size // heads
        if self.head_dim % 2 != 0:
            raise ValueError("head_dim must be even for RoPE")

        self.values = nn.Linear(embed_size, embed_size)
        self.keys = nn.Linear(embed_size, embed_size)
        self.queries = nn.Linear(embed_size, embed_size)
        self.fc_out = nn.Linear(embed_size, embed_size)
        self.dropout = nn.Dropout(dropout)
        self.scale = self.head_dim ** 0.5
        self.rope = RotaryEmbedding(self.head_dim, max_position=max_position)
        self.use_sdpa = hasattr(F, "scaled_dot_product_attention")

    def _project(self, x, linear):
        n = x.shape[0]
        return linear(x).reshape(n, -1, self.heads, self.head_dim)

    def _attend(self, q, k, v, mask=None):
        # Convert from [N, T, H, D] to [N, H, T, D] for SDPA.
        qh = q.permute(0, 2, 1, 3)
        kh = k.permute(0, 2, 1, 3)
        vh = v.permute(0, 2, 1, 3)

        if self.use_sdpa:
            attn_mask = None
            if mask is not None:
                attn_mask = mask.to(dtype=torch.bool)
            out = F.scaled_dot_product_attention(
                qh,
                kh,
                vh,
                attn_mask=attn_mask,
                dropout_p=self.dropout.p if self.training else 0.0,
                is_causal=False,
            )
            return out.permute(0, 2, 1, 3)

        energy = torch.einsum("nqhd,nkhd->nhqk", q, k)
        if mask is not None:
            energy = energy.masked_fill(mask == 0, float("-inf"))
        attention = torch.softmax(energy / self.scale, dim=-1)
        attention = self.dropout(attention)
        return torch.einsum("nhql,nlhd->nqhd", attention, v)

    def forward(self, values, keys, query, mask=None, q_offset=0, k_offset=0):
        n = query.shape[0]
        v = self._project(values, self.values)
        k = self._project(keys, self.keys)
        q = self._project(query, self.queries)

        q_cos, q_sin = self.rope.get_cos_sin(q.size(1), q.device, q.dtype, offset=q_offset)
        k_cos, k_sin = self.rope.get_cos_sin(k.size(1), k.device, k.dtype, offset=k_offset)
        q = apply_rope(q, q_cos, q_sin)
        k = apply_rope(k, k_cos, k_sin)

        out = self._attend(q, k, v, mask=mask).reshape(n, -1, self.heads * self.head_dim)
        return self.fc_out(out)

    def forward_with_cache(self, query, key_value, past_k=None, past_v=None, mask=None, use_cache=False, q_offset=0, k_offset=0):
        n = query.shape[0]
        q = self._project(query, self.queries)
        k_new = self._project(key_value, self.keys)
        v_new = self._project(key_value, self.values)

        q_cos, q_sin = self.rope.get_cos_sin(q.size(1), q.device, q.dtype, offset=q_offset)
        k_cos, k_sin = self.rope.get_cos_sin(k_new.size(1), k_new.device, k_new.dtype, offset=k_offset)
        q = apply_rope(q, q_cos, q_sin)
        k_new = apply_rope(k_new, k_cos, k_sin)

        if past_k is not None:
            k = torch.cat([past_k, k_new], dim=1)
            v = torch.cat([past_v, v_new], dim=1)
        else:
            k, v = k_new, v_new

        out = self._attend(q, k, v, mask=mask).reshape(n, -1, self.heads * self.head_dim)
        out = self.fc_out(out)
        next_k, next_v = (k, v) if use_cache else (None, None)
        return out, next_k, next_v


class EncoderBlockV3(nn.Module):
    def __init__(self, embed_size, heads, forward_expansion, dropout=0.1, max_position=4096):
        super().__init__()
        self.attention = MultiHeadAttentionRoPE(embed_size, heads, dropout, max_position=max_position)
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)
        self.dropout = nn.Dropout(dropout)
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_size, forward_expansion * embed_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(forward_expansion * embed_size, embed_size),
        )

    def forward(self, x, src_mask):
        attn = self.attention(x, x, x, src_mask)
        x = self.norm1(x + self.dropout(attn))
        ff = self.feed_forward(x)
        return self.norm2(x + self.dropout(ff))


class DecoderBlockV3(nn.Module):
    def __init__(self, embed_size, heads, forward_expansion, dropout=0.1, max_position=4096):
        super().__init__()
        self.self_attention = MultiHeadAttentionRoPE(embed_size, heads, dropout, max_position=max_position)
        self.cross_attention = MultiHeadAttentionRoPE(embed_size, heads, dropout, max_position=max_position)
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)
        self.norm3 = nn.LayerNorm(embed_size)
        self.dropout = nn.Dropout(dropout)
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_size, forward_expansion * embed_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(forward_expansion * embed_size, embed_size),
        )

    def forward(self, x, enc_out, src_mask, tgt_mask):
        self_attn = self.self_attention(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(self_attn))
        cross_attn = self.cross_attention(enc_out, enc_out, x, src_mask)
        x = self.norm2(x + self.dropout(cross_attn))
        ff = self.feed_forward(x)
        return self.norm3(x + self.dropout(ff))

    def forward_step(self, x_step, enc_out, src_mask, cache=None, use_cache=True, step_idx=0):
        cache = cache or {}
        self_attn, k_new, v_new = self.self_attention.forward_with_cache(
            query=x_step,
            key_value=x_step,
            past_k=cache.get("self_k"),
            past_v=cache.get("self_v"),
            mask=None,
            use_cache=use_cache,
            q_offset=step_idx,
            k_offset=step_idx,
        )
        x = self.norm1(x_step + self.dropout(self_attn))
        cross_attn = self.cross_attention(enc_out, enc_out, x, src_mask, q_offset=step_idx, k_offset=0)
        x = self.norm2(x + self.dropout(cross_attn))
        ff = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff))
        next_cache = {"self_k": k_new, "self_v": v_new} if use_cache else None
        return x, next_cache


class Seq2SeqTransformerV3(nn.Module):
    def __init__(self, vocab_size, embed_size, num_layers, heads, forward_expansion, max_len, pad_idx, dropout=0.1):
        super().__init__()
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.embed_dropout = nn.Dropout(dropout)
        self.encoder = nn.ModuleList(
            [EncoderBlockV3(embed_size, heads, forward_expansion, dropout, max_position=max_len * 2) for _ in range(num_layers)]
        )
        self.decoder = nn.ModuleList(
            [DecoderBlockV3(embed_size, heads, forward_expansion, dropout, max_position=max_len * 2) for _ in range(num_layers)]
        )
        self.fc_out = nn.Linear(embed_size, vocab_size)

    def make_src_mask(self, src):
        return (src != self.pad_idx).unsqueeze(1).unsqueeze(2)

    def make_tgt_mask(self, tgt):
        b, t = tgt.shape
        causal = torch.tril(torch.ones((t, t), device=tgt.device)).bool().unsqueeze(0).unsqueeze(0)
        tgt_pad = (tgt != self.pad_idx).unsqueeze(1).unsqueeze(2)
        return causal & tgt_pad

    def encode(self, src):
        src_mask = self.make_src_mask(src)
        enc = self.embed_dropout(self.embedding(src))
        for layer in self.encoder:
            enc = layer(enc, src_mask)
        return enc, src_mask

    def decode(self, tgt, enc_out, src_mask):
        tgt_mask = self.make_tgt_mask(tgt)
        dec = self.embed_dropout(self.embedding(tgt))
        for layer in self.decoder:
            dec = layer(dec, enc_out, src_mask, tgt_mask)
        return self.fc_out(dec)

    def forward(self, src, tgt):
        enc_out, src_mask = self.encode(src)
        return self.decode(tgt, enc_out, src_mask)

    def decode_step(self, last_token, enc_out, src_mask, layer_caches, step_idx):
        x = self.embed_dropout(self.embedding(last_token))
        new_caches = []
        for i, layer in enumerate(self.decoder):
            x, next_cache = layer.forward_step(
                x,
                enc_out,
                src_mask,
                cache=layer_caches[i] if layer_caches is not None else None,
                use_cache=True,
                step_idx=step_idx,
            )
            new_caches.append(next_cache)
        logits = self.fc_out(x)
        return logits, new_caches

In [ ]:
@torch.no_grad()
def generate_translation_v2(
    model,
    sentence,
    max_new_tokens=60,
    use_kv_cache=True,
):
    model.eval()

    src_ids = sp.encode(sentence, out_type=int)[: config["max_len"]]
    src = torch.tensor(src_ids, dtype=torch.long, device=device).unsqueeze(0)

    enc_out, src_mask = model.encode(src)
    tgt = torch.tensor([[BOS_ID]], dtype=torch.long, device=device)

    layer_caches = [None for _ in model.decoder] if use_kv_cache else None

    for step in range(max_new_tokens):
        if use_kv_cache:
            logits, layer_caches = model.decode_step(
                last_token=tgt[:, -1:],
                enc_out=enc_out,
                src_mask=src_mask,
                layer_caches=layer_caches,
                step_idx=tgt.size(1) - 1,
            )
            logits = logits[:, -1, :]
        else:
            logits = model.decode(tgt, enc_out, src_mask)[:, -1, :]

        next_id = torch.argmax(logits, dim=-1, keepdim=True)

        tgt = torch.cat([tgt, next_id], dim=1)


        if next_id.item() == EOS_ID or tgt.size(1) >= config["max_len"]:
            break

    pred_ids = tgt.squeeze(0).tolist()
    pred_ids = [i for i in pred_ids if i not in (BOS_ID, EOS_ID, PAD_ID)]
    return sp.decode(pred_ids)


@torch.no_grad()
def beam_search_translation(model, sentence, beam_size=4, max_new_tokens=60, length_penalty=0.7):
    model.eval()

    src_ids = sp.encode(sentence, out_type=int)[: config["max_len"]]
    src = torch.tensor(src_ids, dtype=torch.long, device=device).unsqueeze(0)
    enc_out, src_mask = model.encode(src)

    beams = [(torch.tensor([[BOS_ID]], device=device), 0.0, [None for _ in model.decoder])]
    completed = []

    for _ in range(max_new_tokens):
        all_candidates = []

        for tokens, score, caches in beams:
            if tokens[0, -1].item() == EOS_ID:
                completed.append((tokens, score))
                continue

            logits, next_caches = model.decode_step(
                last_token=tokens[:, -1:],
                enc_out=enc_out,
                src_mask=src_mask,
                layer_caches=caches,
                step_idx=tokens.size(1) - 1,
            )
            log_probs = torch.log_softmax(logits[:, -1, :], dim=-1)
            vals, idxs = torch.topk(log_probs, k=beam_size, dim=-1)

            for j in range(beam_size):
                tok = idxs[0, j].view(1, 1)
                tok_score = vals[0, j].item()
                new_tokens = torch.cat([tokens, tok], dim=1)
                cloned_cache = []
                for cache in next_caches:
                    cloned_cache.append(
                        {
                            "self_k": cache["self_k"].clone(),
                            "self_v": cache["self_v"].clone(),
                        }
                    )
                all_candidates.append((new_tokens, score + tok_score, cloned_cache))

        if not all_candidates:
            break

        all_candidates.sort(
            key=lambda x: x[1] / ((x[0].size(1) ** length_penalty) if x[0].size(1) > 0 else 1.0),
            reverse=True,
        )
        beams = all_candidates[:beam_size]

        if all(b[0][0, -1].item() == EOS_ID for b in beams):
            completed.extend((b[0], b[1]) for b in beams)
            break

    if not completed:
        completed = [(beams[0][0], beams[0][1])]

    completed.sort(
        key=lambda x: x[1] / ((x[0].size(1) ** length_penalty) if x[0].size(1) > 0 else 1.0),
        reverse=True,
    )
    best_tokens = completed[0][0].squeeze(0).tolist()

    out_ids = [i for i in best_tokens if i not in (BOS_ID, EOS_ID, PAD_ID)]
    return sp.decode(out_ids)


model_v2 = Seq2SeqTransformerV3(
    vocab_size=sp.vocab_size(),
    embed_size=config["embed_size"],
    num_layers=config["num_layers"],
    heads=config["heads"],
    forward_expansion=config["forward_expansion"],
    max_len=config["max_len"],
    pad_idx=PAD_ID,
    dropout=config["dropout"],
).to(device)

optimizer = torch.optim.AdamW(model_v2.parameters(), lr=3e-4, betas=(0.9, 0.98), eps=1e-9)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=0.1)

use_amp = torch.cuda.is_available()
scaler = torch.amp.GradScaler(enabled=use_amp) # Updated line


#def lr_lambda(step):
 #   step = max(step, 1)
  #  d_model = config["embed_size"]
  #  warmup = config["warmup_steps"]
  #  return (d_model ** -0.5) * min(step ** -0.5, step * (warmup ** -1.5))

 #scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
lr_now = optimizer.param_groups[0]["lr"]

def run_epoch(model, loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0

    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        decoder_in = tgt[:, :-1]
        target_out = tgt[:, 1:]

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type='cuda', enabled=use_amp): # Updated line
            logits = model(src, decoder_in)
            loss = criterion(logits.reshape(-1, logits.size(-1)), target_out.reshape(-1))

        if train:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
            scaler.step(optimizer)
            scaler.update()
            #scheduler.step()

        total_loss += loss.item()

    return total_loss / max(1, len(loader))


def save_checkpoint(path, epoch, model, optimizer, scaler, best_val, cfg):
    payload = {
        "epoch": epoch,
        "best_val": best_val,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
       # "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "config": cfg,
    }
    torch.save(payload, path)


def maybe_load_checkpoint(path, model, optimizer, scaler):
    if not os.path.exists(path):
        return 0, float("inf")

    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model"])
    #optimizer.load_state_dict(ckpt["optimizer"])
   # scheduler.load_state_dict(ckpt["scheduler"])
    scaler.load_state_dict(ckpt["scaler"])
    print(f"Resumed from epoch {ckpt['epoch'] + 1}")
    return ckpt["epoch"] + 1, ckpt.get("best_val", float("inf"))


start_epoch, best_val = maybe_load_checkpoint(
    config["checkpoint_path"], model_v2, optimizer, scaler
)

metrics = []
bad_epochs = 0
patience = config["early_stopping_patience"]

for epoch in range(start_epoch, config["num_epochs"]):
    train_loss = run_epoch(model_v2, train_loader, train=True)
    val_loss = run_epoch(model_v2, val_loader, train=False)
    lr_now = optimizer.param_groups[0]["lr"]

    # ---- sample prediction debug ----

    sample = "How are you today?"
    pred = generate_translation_v2(model_v2, sample)

    print("INPUT :", sample)
    print("OUTPUT:", pred)
    # --------------------------------

    improved = val_loss < best_val
    if improved:
        best_val = val_loss
        bad_epochs = 0
        save_checkpoint(
            config["checkpoint_path"],
            epoch,
            model_v2,
            optimizer,
            scaler,
            best_val,
            config,
        )
    else:
        bad_epochs += 1

    metrics.append(
        {
            "epoch": epoch + 1,
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
            "lr": float(lr_now),
            "improved": int(improved),
            "bad_epochs": bad_epochs,
        }
    )

    print(
        f"Epoch {epoch+1:02d}/{config['num_epochs']} | "
        f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
        f"lr={lr_now:.6e} | bad_epochs={bad_epochs}/{patience}"
    )

    if bad_epochs >= patience:
        print(f"Early stopping at epoch {epoch+1} (no val improvement for {patience} epochs)")
        break

with open(config["metrics_path"], "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["epoch", "train_loss", "val_loss", "lr", "improved", "bad_epochs"],
    )
    writer.writeheader()
    writer.writerows(metrics)

print("Best val loss:", best_val)
print("Saved metrics to:", config["metrics_path"])

Resumed from epoch 9
INPUT : How are you today?
OUTPUT: Mais que vous est-vous que vous pour vous prévoir?
Epoch 10/50 | train_loss=3.2268 | val_loss=4.4857 | lr=3.000000e-04 | bad_epochs=1/8
INPUT : How are you today?
OUTPUT: Comment vous ure-vous vous avez vous à vous?
Epoch 11/50 | train_loss=3.0088 | val_loss=4.5005 | lr=3.000000e-04 | bad_epochs=2/8
INPUT : How are you today?
OUTPUT: Comment vous ure agroalimentaires?
Epoch 12/50 | train_loss=2.8333 | val_loss=4.5153 | lr=3.000000e-04 | bad_epochs=3/8
INPUT : How are you today?
OUTPUT: Quelle est-il là pour vous aveer l'accès à vous?
Epoch 13/50 | train_loss=2.6817 | val_loss=4.5500 | lr=3.000000e-04 | bad_epochs=4/8
INPUT : How are you today?
OUTPUT: Quels sont vous donnés pour vous pour vous prévoir?
Epoch 14/50 | train_loss=2.5451 | val_loss=4.5846 | lr=3.000000e-04 | bad_epochs=5/8
INPUT : How are you today?
OUTPUT: Comment vous êtes?
Epoch 15/50 | train_loss=2.4300 | val_loss=4.6345 | lr=3.000000e-04 | bad_epochs=6/8
INPUT : 

In [ ]:
@torch.no_grad()
def generate_translation_v2(
    model,
    sentence,
    max_new_tokens=60,
    use_kv_cache=True,
):
    model.eval()

    src_ids = sp.encode(sentence, out_type=int)[: config["max_len"]]
    src = torch.tensor(src_ids, dtype=torch.long, device=device).unsqueeze(0)

    enc_out, src_mask = model.encode(src)
    tgt = torch.tensor([[BOS_ID]], dtype=torch.long, device=device)

    layer_caches = [None for _ in model.decoder] if use_kv_cache else None

    for step in range(max_new_tokens):
        if use_kv_cache:
            logits, layer_caches = model.decode_step(
                last_token=tgt[:, -1:],
                enc_out=enc_out,
                src_mask=src_mask,
                layer_caches=layer_caches,
                step_idx=tgt.size(1) - 1,
            )
            logits = logits[:, -1, :]
        else:
            logits = model.decode(tgt, enc_out, src_mask)[:, -1, :]

        next_id = torch.argmax(logits, dim=-1, keepdim=True)

        tgt = torch.cat([tgt, next_id], dim=1)


        if next_id.item() == EOS_ID or tgt.size(1) >= config["max_len"]:
            break

    pred_ids = tgt.squeeze(0).tolist()
    pred_ids = [i for i in pred_ids if i not in (BOS_ID, EOS_ID, PAD_ID)]
    return sp.decode(pred_ids)


@torch.no_grad()
def beam_search_translation(model, sentence, beam_size=4, max_new_tokens=60, length_penalty=0.7):
    model.eval()

    src_ids = sp.encode(sentence, out_type=int)[: config["max_len"]]
    src = torch.tensor(src_ids, dtype=torch.long, device=device).unsqueeze(0)
    enc_out, src_mask = model.encode(src)

    beams = [(torch.tensor([[BOS_ID]], device=device), 0.0, [None for _ in model.decoder])]
    completed = []

    for _ in range(max_new_tokens):
        all_candidates = []

        for tokens, score, caches in beams:
            if tokens[0, -1].item() == EOS_ID:
                completed.append((tokens, score))
                continue

            logits, next_caches = model.decode_step(
                last_token=tokens[:, -1:],
                enc_out=enc_out,
                src_mask=src_mask,
                layer_caches=caches,
                step_idx=tokens.size(1) - 1,
            )
            log_probs = torch.log_softmax(logits[:, -1, :], dim=-1)
            vals, idxs = torch.topk(log_probs, k=beam_size, dim=-1)

            for j in range(beam_size):
                tok = idxs[0, j].view(1, 1)
                tok_score = vals[0, j].item()
                new_tokens = torch.cat([tokens, tok], dim=1)
                cloned_cache = []
                for cache in next_caches:
                    cloned_cache.append(
                        {
                            "self_k": cache["self_k"].clone(),
                            "self_v": cache["self_v"].clone(),
                        }
                    )
                all_candidates.append((new_tokens, score + tok_score, cloned_cache))

        if not all_candidates:
            break

        all_candidates.sort(
            key=lambda x: x[1] / ((x[0].size(1) ** length_penalty) if x[0].size(1) > 0 else 1.0),
            reverse=True,
        )
        beams = all_candidates[:beam_size]

        if all(b[0][0, -1].item() == EOS_ID for b in beams):
            completed.extend((b[0], b[1]) for b in beams)
            break

    if not completed:
        completed = [(beams[0][0], beams[0][1])]

    completed.sort(
        key=lambda x: x[1] / ((x[0].size(1) ** length_penalty) if x[0].size(1) > 0 else 1.0),
        reverse=True,
    )
    best_tokens = completed[0][0].squeeze(0).tolist()

    out_ids = [i for i in best_tokens if i not in (BOS_ID, EOS_ID, PAD_ID)]
    return sp.decode(out_ids)


print("Sample (KV cache + greedy):", generate_translation_v2(model_v2, "hello, how are you today?"))
print("Sample (beam search):", beam_search_translation(model_v2, "hello, how are you today?"))

Sample (KV cache + greedy): Les exportations du Canada vers la Barbade sont-vous intéressées pour la concurrence?
Sample (beam search): Les exportations du Canada vers la Barbade sont-vous intéressées à la concurrence?


In [ ]:
# Full held-out evaluation: BLEU/chrF + perplexity + inference speed
try:
    import sacrebleu
except ImportError:
    import subprocess

    subprocess.check_call(["pip", "install", "-q", "sacrebleu"])
    import sacrebleu

if os.path.exists(config["checkpoint_path"]):
    best_ckpt = torch.load(config["checkpoint_path"], map_location=device)
    model_v2.load_state_dict(best_ckpt["model"])

model_v2.eval()
print("RoPE attention using SDPA:", getattr(model_v2.decoder[0].self_attention, "use_sdpa", False))

# 1) Translation quality metrics
hypotheses = []
references = []

for _, row in test_df.iterrows():
    hyp = beam_search_translation(model_v2, row["en"], beam_size=4, max_new_tokens=60)
    hypotheses.append(hyp)
    references.append(row["fr"])

bleu = sacrebleu.corpus_bleu(hypotheses, [references])
chrf = sacrebleu.corpus_chrf(hypotheses, [references])

# 2) Teacher-forced perplexity on test set
nll_loss = nn.CrossEntropyLoss(ignore_index=PAD_ID)
test_loss_sum = 0.0
for src, tgt in test_loader:
    src, tgt = src.to(device), tgt.to(device)
    logits = model_v2(src, tgt[:, :-1])
    loss = nll_loss(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))
    test_loss_sum += loss.item()

test_loss = test_loss_sum / max(1, len(test_loader))
test_ppl = math.exp(min(test_loss, 20.0))

# 3) Inference latency + tokens/sec (beam search)
bench_n = min(100, len(test_df))
bench_df = test_df.head(bench_n)
start_t = time.perf_counter()
bench_tokens = 0
for _, row in bench_df.iterrows():
    hyp = beam_search_translation(model_v2, row["en"], beam_size=4, max_new_tokens=60)
    bench_tokens += len(sp.encode(hyp, out_type=int))
elapsed = max(time.perf_counter() - start_t, 1e-9)
latency_ms = (elapsed / bench_n) * 1000.0
toks_per_sec = bench_tokens / elapsed

print(f"Full test size: {len(test_df)}")
print(f"SacreBLEU: {bleu.score:.2f}")
print(f"chrF++: {chrf.score:.2f}")
print(f"Test loss: {test_loss:.4f}")
print(f"Perplexity: {test_ppl:.2f}")
print(f"Latency (beam, per sentence): {latency_ms:.2f} ms")
print(f"Throughput (beam): {toks_per_sec:.2f} tokens/s")

RoPE attention using SDPA: True
Full test size: 1636
SacreBLEU: 12.41
chrF++: 28.43
Test loss: 3.6310
Perplexity: 37.75
Latency (beam, per sentence): 1371.79 ms
Throughput (beam): 9.72 tokens/s
